In [ ]:
from ase.io import read, write

# Took first and last snapshots from training data, scaled their volumes and optimised them using the MLIP
# Following the tutorial: https://jageo.github.io/Advanced_Jobflow_Tutorial/tutorial.html

first_EOS_dft = read('../data/First_EOS.xyz', ':') 
last_EOS_dft = read('../data/Last_EOS.xyz', ':')
first_EOS_nequip = read('../data/first_optimized_all.xyz', ':')
last_EOS_nequip = read('../data/last_optimized_all.xyz', ':')

In [ ]:
from ase.optimize import BFGS
from ase.filters import ExpCellFilter

def get_volume_curve(atoms, calc, range, dir):
    volumes = []
    energies = []

    for x in range:
        optimized_atoms = atoms.copy()
        optimized_atoms.set_cell(atoms.cell * x, scale_atoms=True)
        optimized_atoms.calc = calc
        opt = BFGS(ExpCellFilter(optimized_atoms, constant_volume=True), logfile=f"{dir}/optimization.log")
        opt.run(fmax=0.01)
        write(f'{dir}/optimized_{x}.xyz', optimized_atoms)
        volumes.append(optimized_atoms.get_volume())
        energies.append(optimized_atoms.get_potential_energy())

    return volumes, energies

In [ ]:
from nequip.ase import NequIPCalculator

calculator = NequIPCalculator.from_deployed_model(model_path="../data/NequIP_models/deployed_model_large.pth", device='cuda')

In [ ]:
first_volumes, first_energies = get_volume_curve(atoms[0], calculator, range=[0.95, 0.96, 0.97, 0.98, 0.99, 1.00, 1.01, 1.02, 1.03, 1.04, 1.05], dir='first')

In [ ]:
last_volumes, last_energies = get_volume_curve(atoms[-1], calculator, range=[0.90, 0.91, 0.92, 0.93, 0.94, 0.95, 0.96, 0.97, 0.98, 0.99, 1.00, 1.01, 1.02, 1.03, 1.04, 1.05], dir='last')

In [ ]:
from ase.units import kJ
from ase.eos import EquationOfState
import numpy as np

first_volumes_dft = [first_EOS_dft[i].get_volume() for i in range(len(first_EOS_dft))]
first_energies_dft = [first_EOS_dft[i].info['dft_energy'] for i in range(len(first_EOS_dft))]

sorted_first_dft = np.array(sorted(zip(first_volumes_dft, first_energies_dft), key=lambda x: x[0]))

first_volumes_nequip = [first_EOS_nequip[i].get_volume() for i in range(len(first_EOS_nequip))]
first_energies_nequip = [first_EOS_nequip[i].get_potential_energy() for i in range(len(first_EOS_nequip))]
eos_first_dft = EquationOfState(first_volumes_dft, first_energies_dft-np.max(first_energies_dft))
eos_first_nequip = EquationOfState(first_volumes_nequip, first_energies_nequip-np.max(first_energies_nequip))
v0_dft, e0_dft, B_dft = eos_first_dft.fit()
v0_nequip, e0_nequip, B_nequip = eos_first_nequip.fit()
print(B_dft, 'eV Angstrom**(-3)')
print(B_dft / kJ * 1.0e24, 'GPa')
print(B_nequip, 'eV Angstrom**(-3)')
print(B_nequip / kJ * 1.0e24, 'GPa')

In [ ]:
last_volumes_dft = [last_EOS_dft[i].get_volume() for i in range(len(last_EOS_dft))]
last_energies_dft = [last_EOS_dft[i].info['dft_energy'] for i in range(len(last_EOS_dft))]

sorted_last_dft = np.array(sorted(zip(last_volumes_dft, last_energies_dft), key=lambda x: x[0]))

last_volumes_nequip = [last_EOS_nequip[i].get_volume() for i in range(len(last_EOS_nequip))]
last_energies_nequip = [last_EOS_nequip[i].get_potential_energy() for i in range(len(last_EOS_nequip))]

# disregard the first three and last two points, which have different tilts than original

eos_last_dft = EquationOfState(sorted_last_dft[:, 0][3:-2], sorted_last_dft[:, 1][3:-2]-np.max(sorted_last_dft[:, 1][3:-2]))
eos_last_nequip = EquationOfState(last_volumes_nequip[3:-2], last_energies_nequip[3:-2]-np.max(last_energies_nequip[3:-2]))
v0_dft, e0_dft, B_dft = eos_last_dft.fit()
v0_nequip, e0_nequip, B_nequip = eos_last_nequip.fit()
print(B_dft, 'eV Angstrom**(-3)')
print(B_dft / kJ * 1.0e24, 'GPa')
print(B_nequip, 'eV Angstrom**(-3)')
print(B_nequip / kJ * 1.0e24, 'GPa')

In [ ]:
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
from matplotlib import rcParams
import numpy as np

rcParams['font.family'] = "ARIAL"

fig, axs = plt.subplots(1, 2, figsize=(17/2.54, 7.5/2.54), dpi=300)

axs[0].scatter(eos_first_nequip.v, eos_first_nequip.e, label='NequIP', color='r')
axs[0].plot(eos_first_nequip.getplotdata()[4], eos_first_nequip.getplotdata()[5], '--', color='r')
axs[0].scatter(eos_first_dft.v, eos_first_dft.e, label='DFT', color='black')
axs[0].plot(eos_first_dft.getplotdata()[4], eos_first_dft.getplotdata()[5], '--', color='black')
axs[0].set_title('low-T EOS', fontsize=9)


axs[1].scatter(eos_last_nequip.v, eos_last_nequip.e, label='NequIP', color='r')
axs[1].plot(eos_last_nequip.getplotdata()[4], eos_last_nequip.getplotdata()[5], '--', color='r')
axs[1].scatter(eos_last_dft.v, eos_last_dft.e, label='DFT', color='black')
axs[1].plot(eos_last_dft.getplotdata()[4], eos_last_dft.getplotdata()[5], '--', color='black')
axs[1].set_title('high-T EOS', fontsize=9)


for ax in axs:
    for axis in ['top', 'bottom', 'left', 'right']:
        ax.spines[axis].set_linewidth(1)
    ax.tick_params(width=1, labelsize=7, length=3)

    # set font sizes to 9
    ax.xaxis.label.set_size(9)
    ax.yaxis.label.set_size(9)

    ax.set_xlabel(r'Volume ($\mathrm{\AA}^3$)', fontsize=9)
    ax.set_ylabel('Relative energy (eV)', fontsize=9)

plt.legend(frameon=False, fontsize=7, loc='upper right')
plt.tight_layout()
plt.savefig('eos_comparison.png')